In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt

## Functions

In [2]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

## Constants 

In [3]:
# project
str_project = os.getcwd().split('\\')[3].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[4]
print(f'Task: {str_task}')
# output
str_subtask = os.getcwd().split('\\')[5]
print(f'Subtask: {str_subtask}')
str_dirname_output = './output'

Project: 20240521-bridger-internship
Task: 13_payload_parsing
Subtask: 01_pull_data


In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

## Read SQL

In [5]:
str_filepath = './sql/script.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('select \n'
 '\ttbltempstaticpool.bigAccountId,\n'
 '\ttbltempstaticpool.dtmFunded,\n'
 '\ttblDove.strRequest\n'
 'from riskdb.analytics.tbltempstaticpool\n'
 'left outer join\n'
 '(\n'
 '\tselect\n'
 '\t\tbigAccountId,\n'
 '\t\tmax(bigDoveId) as bigDoveId\n'
 '\tfrom zestdb.dbo.tblDove\n'
 '\tgroup by bigAccountId\n'
 ')tblMax1 on tblMax1.bigAccountid=tbltempstaticpool.bigAccountId\n'
 'left outer join zestdb.dbo.tblDove on tblDove.bigDoveId=tblMax1.bigDoveId\n'
 "where dtmFunded >= '01/01/2023'")


## Pull data

In [6]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# rm None requests
df = df[df['strRequest'].notna()]

# show
df

<timed exec>:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


CPU times: total: 56.5 s
Wall time: 12min 55s


,bigAccountId,dtmFunded,strRequest
3696,6508394,2023-01-03,"{""request_id"":""6508394192828"",""rows"":[{""row_id..."
3713,6466527,2023-01-03,"{""request_id"":""6466527957485"",""rows"":[{""row_id..."
3732,6485230,2023-01-03,"{""request_id"":""6485230420313"",""rows"":[{""row_id..."
3767,6481300,2023-01-03,"{""request_id"":""6481300782840"",""rows"":[{""row_id..."
3776,6517284,2023-01-03,"{""request_id"":""65172847073"",""rows"":[{""row_id"":..."
...,...,...,...
36448,7280119,2023-12-08,"{""request_id"":""7280119797671"",""rows"":[{""row_id..."
36449,7362606,2023-12-04,"{""request_id"":""7362606941531"",""rows"":[{""row_id..."
36450,7281439,2023-11-27,"{""request_id"":""7281439715885"",""rows"":[{""row_id..."
36451,7373007,2023-12-13,"{""request_id"":""7373007690715"",""rows"":[{""row_id..."


### Keep Recent, Drop Duplicates

In [7]:
df.drop_duplicates(
    subset=['bigAccountId'],
    keep='last', 
    inplace=True,
)
# show
df

,bigAccountId,dtmFunded,strRequest
3696,6508394,2023-01-03,"{""request_id"":""6508394192828"",""rows"":[{""row_id..."
3713,6466527,2023-01-03,"{""request_id"":""6466527957485"",""rows"":[{""row_id..."
3732,6485230,2023-01-03,"{""request_id"":""6485230420313"",""rows"":[{""row_id..."
3767,6481300,2023-01-03,"{""request_id"":""6481300782840"",""rows"":[{""row_id..."
3776,6517284,2023-01-03,"{""request_id"":""65172847073"",""rows"":[{""row_id"":..."
...,...,...,...
36448,7280119,2023-12-08,"{""request_id"":""7280119797671"",""rows"":[{""row_id..."
36449,7362606,2023-12-04,"{""request_id"":""7362606941531"",""rows"":[{""row_id..."
36450,7281439,2023-11-27,"{""request_id"":""7281439715885"",""rows"":[{""row_id..."
36451,7373007,2023-12-13,"{""request_id"":""7373007690715"",""rows"":[{""row_id..."


### Save and Upload

In [8]:
%%time 

# save
str_filename = 'df_requests.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

ArrowMemoryError: realloc of size 3742892032 failed

In [9]:
# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_filename}', 
    str_bucket_name=str_project,
)

FileNotFoundError: [WinError 2] The system cannot find the file specified: './output/df_requests.gzip'